In [1]:
from click import DateTime
# allow to sync with local code
%load_ext autoreload
%autoreload 2

In [15]:

from datetime import datetime

EXPERIMENT_IDS = {
    "random": {
        "CIFAR_10 - 10%": None,
        "Pascal_VOC - 10%": None,
        "Imdb - 10%": None,
    },
    "gold": {
        "CIFAR_10 - 10%": "354041301038171263",
        "Pascal_VOC - 10%": "251710914009724832",
        "Imdb - 10%": "595772664053431657",
    }
}
EXPERIMENT_DATES = {
    "random": {
        "CIFAR_10 - 10%": None,
        "Pascal_VOC - 10%": None,
        "Imdb - 10%": None,
    },
    "gold": {
        # "CIFAR_10 - 10%": {
        #     "start": datetime(2026, 5, 16, 7, 0, 0),
        #     "end": datetime(2026, 5, 16, 11, 30, 0),
        # },
        "CIFAR_10 - 10%": {
            "start": datetime(2026, 5, 29, 15, 0, 0),
            "end": datetime(2026, 5, 29, 19, 30, 0),
        },
        "Pascal_VOC - 10%": {
            "start": datetime(2026, 5, 22, 15, 44, 0),
            "end": datetime(2026, 5, 23, 3, 30, 0),
        },
        # "Pascal_VOC - 10%": {
        #     "start": datetime(2026, 5, 21, 21, 0, 0),
        #     "end": datetime(2026, 5, 22, 8, 0, 0),
        # },
        # "Imdb - 10%": {
        #     "start": datetime(2026, 5, 20, 16, 30, 0),
        #     "end": datetime(2026, 5, 21, 4, 30, 0),
        # },
        "Imdb - 10%": {
            "start": datetime(2026, 5, 18, 8, 30, 0),
            "end": datetime(2026, 5, 19, 7, 30, 0),
        },
    }
}


METRICS = ("test_auroc", "test_acc", "test_iou", "test_micro_iou")

MLFLOW_TRACKING_URI = "/home/ubuntu/goldener-examples/mlruns/"


In [16]:
from mlflow import MlflowClient

client = MlflowClient(
    tracking_uri=MLFLOW_TRACKING_URI
)



def get_metric_for_experiment(experiment_id, metric_names):
    """
    Get run name and specific metric value for all runs in an experiment.
    """
    runs = client.search_runs(experiment_ids=[experiment_id])

    run_data = []
    for run in runs:
        run_name = run.info.run_name
        run_date = datetime.fromtimestamp(run.info.start_time / 1000)
        random_shuffle_state = run.data.params["random_shuffle_state"]
        run_info = {"run_name": run_name, "seed": random_shuffle_state, "starts": run_date}
        metrics_info = {}
        for metric_name in metric_names:
            metric_value = run.data.metrics.get(metric_name, None)
            if metric_value is not None:
                metrics_info[metric_name] = metric_value
        if metrics_info:
            run_data.append(run_info | metrics_info)

    return run_data

In [17]:
# --- Collect all data for all datasets ---
all_data = {}
for shuffling_type, experiment_ids in EXPERIMENT_IDS.items():
    all_data[shuffling_type] = {}
    for dataset_name, experiment_id in experiment_ids.items():
        if experiment_id is not None:
            start = EXPERIMENT_DATES[shuffling_type][dataset_name]["start"]
            end = EXPERIMENT_DATES[shuffling_type][dataset_name]["end"]
            run_metrics = get_metric_for_experiment(
                experiment_id,
                METRICS,
            )
            run_metrics = [
                metrics
                for metrics in run_metrics
                if start <= metrics["starts"] < end
            ]
            if len(run_metrics) != 5:
                raise ValueError(f"too many or not enough runs: {len(run_metrics)} runs")

            all_data[shuffling_type][dataset_name] = run_metrics

all_data

{'random': {},
 'gold': {'CIFAR_10 - 10%': [{'run_name': 'cifar10_gold_resnet_42_86',
    'seed': '86',
    'starts': datetime.datetime(2026, 5, 29, 17, 14, 18, 294000),
    'test_auroc': 0.9905418753623962,
    'test_acc': 0.8838000297546387},
   {'run_name': 'cifar10_gold_resnet_42_75',
    'seed': '75',
    'starts': datetime.datetime(2026, 5, 29, 16, 47, 27, 956000),
    'test_auroc': 0.9893986582756042,
    'test_acc': 0.8748999834060669},
   {'run_name': 'cifar10_gold_resnet_42_64',
    'seed': '64',
    'starts': datetime.datetime(2026, 5, 29, 16, 20, 34, 703000),
    'test_auroc': 0.9917212724685669,
    'test_acc': 0.8826000094413757},
   {'run_name': 'cifar10_gold_resnet_42_53',
    'seed': '53',
    'starts': datetime.datetime(2026, 5, 29, 15, 53, 39, 659000),
    'test_auroc': 0.9904713034629822,
    'test_acc': 0.8772000074386597},
   {'run_name': 'cifar10_gold_resnet_42_42',
    'seed': '42',
    'starts': datetime.datetime(2026, 5, 29, 15, 26, 45, 755000),
    'test_auro

In [18]:
import pandas as pd

# --- Convert to flat DataFrame ---
rows = []
for shuffling_type, datasets in all_data.items():
    for dataset_name, runs in datasets.items():
        for run in runs:
            for metric in METRICS:
                if metric in run:
                    rows.append(
                        {
                            "shuffle": shuffling_type,
                            "dataset": dataset_name,
                            "run_name": run["run_name"],
                            "seed": str(run["seed"]),
                            "metric": metric,
                            "value": run[metric],
                        }
                    )

df = pd.DataFrame(rows)
df.head()

,shuffle,dataset,run_name,seed,metric,value
0,gold,CIFAR_10 - 10%,cifar10_gold_resnet_42_86,86,test_auroc,0.990542
1,gold,CIFAR_10 - 10%,cifar10_gold_resnet_42_86,86,test_acc,0.883800
2,gold,CIFAR_10 - 10%,cifar10_gold_resnet_42_75,75,test_auroc,0.989399
3,gold,CIFAR_10 - 10%,cifar10_gold_resnet_42_75,75,test_acc,0.874900
4,gold,CIFAR_10 - 10%,cifar10_gold_resnet_42_64,64,test_auroc,0.991721


In [19]:
metrics_present = [m for m in METRICS if m in df["metric"].values]

summary = df.groupby(["shuffle", "dataset", "metric"])["value"].agg(mean="mean", std="std").round(4)

# Build one row per dataset with "min / max" string per metric
table_rows = []
for shuffle, experiment_ids in EXPERIMENT_IDS.items():
    for dataset in experiment_ids.keys():
        row = {"shuffle": shuffle, "dataset": dataset}
        for metric in metrics_present:
            mean_metric_row = f"{metric} (mean +/- std)"
            try:
                mean = summary.loc[(shuffle, dataset, metric), "mean"]
                std = summary.loc[(shuffle, dataset, metric), "std"]
                row[mean_metric_row] = f"{mean} +/- {std}"
            except KeyError:
                row[mean_metric_row] = "-"
        table_rows.append(row)

table_df = pd.DataFrame(table_rows).set_index("dataset")
print(table_df.to_markdown())

| dataset          | shuffle   | test_auroc (mean +/- std)   | test_acc (mean +/- std)   | test_iou (mean +/- std)   | test_micro_iou (mean +/- std)   |
|:-----------------|:----------|:----------------------------|:--------------------------|:--------------------------|:--------------------------------|
| CIFAR_10 - 10%   | random    | -                           | -                         | -                         | -                               |
| Pascal_VOC - 10% | random    | -                           | -                         | -                         | -                               |
| Imdb - 10%       | random    | -                           | -                         | -                         | -                               |
| CIFAR_10 - 10%   | gold      | 0.9906 +/- 0.0008           | 0.8798 +/- 0.0037         | -                         | -                               |
| Pascal_VOC - 10% | gold      | -                           | 0.8183 +/- 0.0014  

In [20]:
metrics_present = [m for m in METRICS if m in df["metric"].values]

summary = df.groupby(["shuffle", "dataset", "metric"])["value"].agg(max="max").round(4)

# Build one row per dataset with "min / max" string per metric
table_rows = []
for shuffle, experiment_ids in EXPERIMENT_IDS.items():
    for dataset in experiment_ids.keys():
        row = {"shuffle": shuffle, "dataset": dataset}
        for metric in metrics_present:
            max_metric_row = f"{metric} max"
            try:
                max = summary.loc[(shuffle, dataset, metric), "max"]
                row[max_metric_row] = f"{max}"
            except KeyError:
                row[max_metric_row] = "-"
        table_rows.append(row)

table_df = pd.DataFrame(table_rows).set_index("dataset")
print(table_df.to_markdown())

| dataset          | shuffle   | test_auroc max   | test_acc max   | test_iou max   | test_micro_iou max   |
|:-----------------|:----------|:-----------------|:---------------|:---------------|:---------------------|
| CIFAR_10 - 10%   | random    | -                | -              | -              | -                    |
| Pascal_VOC - 10% | random    | -                | -              | -              | -                    |
| Imdb - 10%       | random    | -                | -              | -              | -                    |
| CIFAR_10 - 10%   | gold      | 0.9917           | 0.8838         | -              | -                    |
| Pascal_VOC - 10% | gold      | -                | 0.8198         | 0.2628         | 0.6946               |
| Imdb - 10%       | gold      | 0.8395           | 0.7662         | -              | -                    |
